# Lesson 6 — Statistical Validation: Granger Causality and Event Studies

**Learning objective:** Understand the two complementary statistical tracks that validate whether a canonical mechanism has predictive power for stock returns. Run both tracks on synthetic data — no live market data or AWS credentials required.

---

## Two tracks, two different statistical powers

```
Track A — Granger (N ≈ 22 quarters)   Low power, illustrative only
  aligned_df
    └─▶ Fix 1: ADF stationarity + auto-differencing
    └─▶ Fix 2: sparsity filter (≥30% non-zero quarters)
    └─▶ Fix 3: BH FDR correction (never raw p-values)
    └─▶ grangercausalitytests()  →  granger_df

Track B — Event Study (N ≈ trading days)   High power
  PTC pool + daily prices
    └─▶ estimation window [-120, -11] trading days
    └─▶ normal return = mean(estimation window)
    └─▶ CAR = Σ(actual - normal) over [+1, +30] trading days
    └─▶ t-test per canonical  →  eventstudy_df
```

**Why two tracks?** Granger tests temporal precedence at the quarterly frequency — it's the right diagnostic for long-horizon themes (e.g. Red Sea disruptions persist 3 quarters). The event study operates at daily frequency and directly measures the market's real-time reaction to a management signal, which is the investment-relevant quantity.

In [ ]:
import sys, json, tempfile
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd

sys.path.insert(0, '../../src')

from signal_intelligence.correlation import (
    _bh_correction, _bonferroni, _is_stationary, _make_stationary,
    run_granger_analysis,
)
from signal_intelligence.eventstudy import compute_car, run_event_study

---

## Track A — Granger Causality

### Fix 1: Stationarity test (ADF) and auto-differencing

Granger causality requires stationary time series. A non-stationary series (random walk) will produce spurious Granger results — a well-known failure mode called **spurious regression**.

The Augmented Dickey-Fuller (ADF) test checks for a unit root:
- p < 0.05 → stationary, use as-is
- p ≥ 0.05 → non-stationary, first-difference the series and re-test

We never manually decide whether to difference — the ADF test decides.

In [ ]:
# Build two synthetic quarterly series: one stationary, one non-stationary
rng = np.random.default_rng(42)
n = 24  # 6 years of quarterly data

# Stationary: mean-reverting around 5
stationary_series = pd.Series(
    5 + rng.normal(0, 1, n),
    index=pd.date_range("2018-03-31", periods=n, freq="QE")
)

# Non-stationary: random walk (no mean reversion)
nonstationary_series = pd.Series(
    np.cumsum(rng.normal(0, 1, n)),
    index=pd.date_range("2018-03-31", periods=n, freq="QE")
)

print("Stationary series — is_stationary():", _is_stationary(stationary_series))
print("Non-stationary series — is_stationary():", _is_stationary(nonstationary_series))

diffed, was_differenced = _make_stationary(nonstationary_series)
print(f"\nAfter _make_stationary(): was_differenced={was_differenced}")
print(f"Differenced series — is_stationary(): {_is_stationary(diffed)}")

### Fix 2: Sparsity filter

Many canonicals appear in only 3–5 quarters out of 22. A Granger test with 85% zeros is meaningless — the model is fitting noise. We require **≥30% non-zero quarters** before testing.

This is not optional: with 100+ canonicals, even random noise will produce ~5 false positives at α=0.05 without any filtering. The sparsity filter reduces the multiple-testing burden and keeps only economically meaningful signals.

In [ ]:
# Illustration: sparsity fraction for two canonicals
n_quarters = 22

# Dense canonical — 12 non-zero quarters out of 22 (54%)
dense = [2, 0, 1, 3, 0, 2, 1, 0, 2, 1, 3, 0, 2, 1, 0, 1, 2, 0, 1, 2, 1, 3]
# Sparse canonical — only 4 mentions in 22 quarters (18%)
sparse = [0] * 18 + [1, 0, 2, 0]

for label, mentions in [("Dense", dense), ("Sparse", sparse)]:
    arr = np.array(mentions)
    n_nonzero = (arr > 0).sum()
    sparsity_pct = n_nonzero / n_quarters * 100
    will_be_tested = sparsity_pct >= 30.0
    print(f"{label:6s}: {n_nonzero}/{n_quarters} non-zero quarters = {sparsity_pct:.0f}% → tested={will_be_tested}")

### Fix 3: Multiple testing correction (Benjamini-Hochberg)

If we test 50 canonicals at α=0.05, we expect ~2.5 false positives by chance alone. Reporting raw p-values is **wrong**.

The Benjamini-Hochberg (BH) procedure controls the **False Discovery Rate (FDR)**: the expected fraction of significant results that are false positives. It is less conservative than Bonferroni (which controls the probability of any false positive) and therefore more appropriate when we have many true positives.

If BH fails, we fall back to Bonferroni (`min(p × n, 1.0)`) — never raw p.

In [ ]:
# Simulate 10 hypothesis tests: 3 truly significant, 7 noise
true_signals = [0.001, 0.003, 0.012]     # real effects (raw p)
noise = [0.08, 0.15, 0.22, 0.41, 0.53, 0.71, 0.88]   # noise
raw_p_values = true_signals + noise

bh_corrected = _bh_correction(raw_p_values, alpha=0.05)
bonferroni_corrected = _bonferroni(raw_p_values)

print(f"{'Raw p':>10}  {'BH-corrected':>14}  {'Bonferroni':>12}  {'BH sig?':>8}")
print("-" * 52)
for raw, bh, bf in zip(raw_p_values, bh_corrected, bonferroni_corrected):
    sig = "✓" if bh < 0.05 else ""
    print(f"{raw:10.4f}  {bh:14.4f}  {bf:12.4f}  {sig:>8}")

print(f"\nBH significant: {sum(p < 0.05 for p in bh_corrected)}/10")
print(f"Bonferroni significant: {sum(p < 0.05 for p in bonferroni_corrected)}/10")
print("\nBH is less conservative — captures more true signals, controls FDR rather than FWER.")

### Exercise 1 — Run the full Granger pipeline

We build a synthetic aligned DataFrame with two canonicals:
- `canon-precursor`: high sparsity → will pass the filter, Granger-tested
- `canon-sparse`: low sparsity → filtered out

Then run `run_granger_analysis()` end-to-end.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Build synthetic quarterly data
rng = np.random.default_rng(0)
quarters = pd.date_range("2018-03-31", periods=24, freq="QE")

# Precursor canonical — mention count leads earnings surprise
mention_counts = np.clip(rng.integers(0, 4, size=24), 0, None).astype(float)
# Earnings surprise is correlated with mentions 1 quarter later (simulated lag-1 lead)
surprise_lagged = np.roll(mention_counts, 1) * 0.5 + rng.normal(0, 0.3, 24)
surprise_lagged[0] = 0.0

rows = []
for i, q in enumerate(quarters):
    rows.append({"canonical_id": "canon-precursor", "quarter_end": q,
                 "mention_count": mention_counts[i], "surprise_pct": surprise_lagged[i]})

# Sparse canonical — only 4 non-zero mentions (too sparse)
sparse_mentions = np.zeros(24)
sparse_mentions[[3, 8, 15, 20]] = 1.0
for i, q in enumerate(quarters):
    rows.append({"canonical_id": "canon-sparse", "quarter_end": q,
                 "mention_count": sparse_mentions[i], "surprise_pct": rng.normal(0, 0.5)})

aligned_df = pd.DataFrame(rows)

# Canonical metadata
canonical_df = pd.DataFrame([
    {"canonical_id": "canon-precursor", "direction": "precursor"},
    {"canonical_id": "canon-sparse",    "direction": "precursor"},
])

with tempfile.TemporaryDirectory() as tmpdir:
    granger_df = run_granger_analysis(
        aligned_df=aligned_df,
        canonical_df=canonical_df,
        output_dir=Path(tmpdir),
        ticker="DEMO",
    )

display_cols = ["canonical_id", "n_quarters", "n_nonzero", "sparsity_pct",
                "filtered_out", "differenced_mention", "best_lag", "p_value_raw",
                "p_value_bh", "significant_bh", "granger_direction"]
print(granger_df[display_cols].to_string(index=False))

---

## Track B — Event Study

### Theory: cumulative abnormal return (CAR)

The event study directly answers: *does the stock move when management discloses a precursor mechanism?*

```
Event date = source_date (document publication, always tradable)
                ┌──────────────────────┐
                │  Estimation window   │       Event window
    ◄───────────┤ [-120, -11] trading  ├─ source_date ─────────────────► day +30
                │  days before event   │           [+1, +30]
                └──────────────────────┘

normal_return = mean(actual returns in estimation window)
abnormal_return(t) = actual_return(t) - normal_return
CAR = Σ abnormal_return over [+1, +30]
```

**Why source_date and not event_date?** The event_date (when the underlying event happened, e.g. Red Sea disruption) may predate the document. Using it would create **look-ahead bias** — we'd be "predicting" the past. The source_date (when the document was published) is the earliest date the market could have reacted.

### Exercise 2 — `compute_car()` on a single event

In [ ]:
# Build 3 years of synthetic daily returns
rng = np.random.default_rng(7)
trading_days = pd.date_range("2022-01-01", "2025-01-01", freq="B")  # business days
returns = pd.Series(
    rng.normal(0.0003, 0.012, size=len(trading_days)),   # ~30bp daily mean, 1.2% vol
    index=trading_days
)

# Event: management press release on 2024-03-15
event_date = pd.Timestamp("2024-03-15")

car = compute_car(event_date, returns)
print(f"Event date       : {event_date.date()}")
print(f"CAR [+1, +30]    : {car:.4f}  ({car*100:.2f}%)")
print()

# Inspect the windows manually
from signal_intelligence.eventstudy import _trading_day_offset
date_idx = pd.DatetimeIndex(returns.index)
est_start = _trading_day_offset(date_idx, event_date, -120)
est_end   = _trading_day_offset(date_idx, event_date, -11)
car_start = _trading_day_offset(date_idx, event_date, 1)
car_end   = _trading_day_offset(date_idx, event_date, 30)

estimation_window = returns.loc[est_start:est_end]
normal_return = estimation_window.mean()
event_window = returns.loc[car_start:car_end]

print(f"Estimation window: {est_start.date()} → {est_end.date()}  ({len(estimation_window)} days)")
print(f"Normal return    : {normal_return:.6f}  ({normal_return*100:.4f}%/day)")
print(f"Event window     : {car_start.date()} → {car_end.date()}  ({len(event_window)} days)")
print(f"Sum actual ret   : {event_window.sum():.4f}")
print(f"Abnormal total   : {(event_window - normal_return).sum():.4f}  ← this is CAR")

### Exercise 3 — Aggregate CAR across multiple events (run_event_study)

In [ ]:
from unittest.mock import patch
from ptc_extraction.models import PTC, Direction

# Five event dates for a precursor canonical
event_dates = [
    pd.Timestamp("2022-11-10"),  pd.Timestamp("2023-02-15"),
    pd.Timestamp("2023-08-03"),  pd.Timestamp("2023-11-14"),
    pd.Timestamp("2024-03-15"),
]
source_ids = [f"ML.PA-mgmt-{d.strftime('%Y-%m-%d')}" for d in event_dates]

# Prices DataFrame (adj_close from our synthetic returns)
prices_df = pd.DataFrame(
    {"adj_close": (1.0 + returns).cumprod() * 100},
    index=returns.index,
)

# Canonical table: one precursor, one consequence
canonical_df = pd.DataFrame([
    {
        "canonical_id": "ML.PA-management-1",
        "direction": "precursor",
        "source_ids": json.dumps(source_ids),
    },
    {
        "canonical_id": "ML.PA-management-2",
        "direction": "consequence",
        "source_ids": json.dumps(source_ids[:2]),
    },
])

# Mock load_pool to return fake PTCs with our source dates
def make_ptc(sid, event_date):
    return PTC(
        mechanism="Red Sea disruptions elevated logistics costs.",
        raw_text="raw",
        direction=Direction.precursor,
        polarity=-1,
        source_id=sid,
        source_date=event_date.date(),
        source_type="management",
        extracted_by="mock-model",
        extraction_version="v1.1",
    )

fake_ptcs = [make_ptc(sid, d) for sid, d in zip(source_ids, event_dates)]

with tempfile.TemporaryDirectory() as tmpdir:
    pool_path = Path(tmpdir) / "pool.jsonl"
    pool_path.write_text("")  # placeholder

    with patch("signal_intelligence.eventstudy.load_pool", return_value=fake_ptcs):
        eventstudy_df = run_event_study(
            canonical_df=canonical_df,
            pool_path=pool_path,
            prices_df=prices_df,
            output_dir=Path(tmpdir),
            ticker="ML.PA",
        )

print(eventstudy_df[["canonical_id", "direction", "n_events",
                     "mean_car", "std_car", "t_stat", "p_value", "positive_pct"]].to_string(index=False))

### What to look for in the event study output

| Column | Interpretation |
|--------|----------------|
| `n_events` | Number of source dates contributing to this canonical. Consequence canonicals are always 0 (market already moved before publication). |
| `mean_car` | Average cumulative abnormal return over [+1,+30] trading days. A positive mean_car for a negative-polarity precursor is suspicious (market disagreement). |
| `t_stat` | How many standard errors the mean_car is from zero. Meaningful once n_events ≥ 10. |
| `p_value` | Two-tailed t-test. Treat as one of several signals, not a binary verdict. |
| `positive_pct` | Fraction of individual events where CAR > 0. A robust canonical should have consistent sign. |

### The five canonical families revisited

Track B distinguishes **Gold** (triangulated, precursor, significant CAR) from the others:

In [ ]:
families = [
    {"Family": "Gold",                 "corpus_presence": "triangulated",    "direction": "precursor",    "Track B": "significant CAR"},
    {"Family": "Privileged",           "corpus_presence": "insider_only",     "direction": "precursor",    "Track B": "significant CAR"},
    {"Family": "Management blind spot","corpus_presence": "insider_only",     "direction": "consequence",  "Track B": "not tested (n=0)"},
    {"Family": "Sector baseline",      "corpus_presence": "triangulated",     "sector_tag": "sector_wide","Track B": "CAR near 0 (priced in)"},
    {"Family": "Narrative inflation",  "corpus_presence": "outsider_only",    "direction": "precursor",    "Track B": "no significant CAR"},
]
print(pd.DataFrame(families).to_string(index=False))

---

## Key takeaways

1. **Fix 1 (ADF)**: Always test stationarity before Granger. Auto-difference non-stationary series — never skip this step.
2. **Fix 2 (sparsity)**: A canonical with <30% non-zero quarters has no temporal structure to test. Remove it from Granger, not from the canonical table.
3. **Fix 3 (BH)**: With O(100) canonicals, use BH FDR correction. Never report raw p-values. Bonferroni is a valid fallback.
4. **Track A has low power** (~22 quarters). Use it as a sanity check, not a primary result.
5. **Track B anchors on source_date** to avoid look-ahead bias. Event studies operate at daily frequency with O(events) > O(quarters).
6. **Consequence canonicals skip Track B**: the impact already happened before the document — no market reaction is expected post-publication.

**Next:** Lesson 7 — Pipeline Orchestration: wiring all phases with `PipelineConfig` and `run_pipeline`.